In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.io as pio
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error , r2_score
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import nltk
import webbrowser
import os

In [2]:
nltk.download('vader_lexicon')

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\hi\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


True

# Loading Data

In [3]:
apps_df = pd.read_csv("play-store-data.csv")
reviews_df = pd.read_csv("user-reviews.csv")

# View DataType

In [4]:
apps_df.dtypes

App                object
Category           object
Rating            float64
Reviews            object
Size               object
Installs           object
Type               object
Price              object
Content Rating     object
Genres             object
Last Updated       object
Current Ver        object
Android Ver        object
dtype: object

In [5]:
reviews_df.dtypes

App                        object
Translated_Review          object
Sentiment                  object
Sentiment_Polarity        float64
Sentiment_Subjectivity    float64
dtype: object

# Cleaning the Data

In [6]:
apps_df = apps_df.dropna(subset = ['Rating'])
for column in apps_df.columns:
    apps_df[column] = apps_df[column].fillna(apps_df[column].mode()[0])
apps_df.drop_duplicates(inplace = True)
apps_df = apps_df[apps_df['Rating']<=5]

reviews_df.dropna(subset = ["Translated_Review"] , inplace = True)

#### Converts Install column to integer by removing commas and '+'

In [7]:
apps_df['Installs'] = apps_df['Installs'].str.replace(',','').str.replace('+','').astype(int)

#### Converts Price column to Integer by removing ''$''

In [8]:
apps_df['Price'] = apps_df['Price'].str.replace('$',"").astype(float)

In [9]:
apps_df["Installs"] = apps_df["Installs"].astype(int)
apps_df["Reviews"] = apps_df["Reviews"].astype(int)

In [10]:
merged_df = pd.merge(apps_df , reviews_df , on = "App" , how = "inner")

# Data Transformation

In [11]:
def convert_size(size):
    if 'M' in size:
        return float(size.replace('M',''))
    elif 'k' in size:
        return float(size.replace('k',''))/1024
    else:
        return np.nan

apps_df['Size'] = apps_df['Size'].apply(convert_size)

In [12]:
apps_df

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,159,19.0,10000,Free,0.0,Everyone,Art & Design,"January 7, 2018",1.0.0,4.0.3 and up
1,Coloring book moana,ART_AND_DESIGN,3.9,967,14.0,500000,Free,0.0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.7,87510,8.7,5000000,Free,0.0,Everyone,Art & Design,"August 1, 2018",1.2.4,4.0.3 and up
3,Sketch - Draw & Paint,ART_AND_DESIGN,4.5,215644,25.0,50000000,Free,0.0,Teen,Art & Design,"June 8, 2018",Varies with device,4.2 and up
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,4.3,967,2.8,100000,Free,0.0,Everyone,Art & Design;Creativity,"June 20, 2018",1.1,4.4 and up
...,...,...,...,...,...,...,...,...,...,...,...,...,...
10834,FR Calculator,FAMILY,4.0,7,2.6,500,Free,0.0,Everyone,Education,"June 18, 2017",1.0.0,4.1 and up
10836,Sya9a Maroc - FR,FAMILY,4.5,38,53.0,5000,Free,0.0,Everyone,Education,"July 25, 2017",1.48,4.1 and up
10837,Fr. Mike Schmitz Audio Teachings,FAMILY,5.0,4,3.6,100,Free,0.0,Everyone,Education,"July 6, 2018",1.0,4.1 and up
10839,The SCP Foundation DB fr nn5n,BOOKS_AND_REFERENCE,4.5,114,NaN,1000,Free,0.0,Mature 17+,Books & Reference,"January 19, 2015",Varies with device,Varies with device


In [13]:
apps_df['Log_Installs'] = np.log1p(apps_df["Installs"])
apps_df['Log_Reviews'] = np.log1p(apps_df["Reviews"])

In [14]:
def rating_group(rating):
    if rating >= 4:
        return "Top rated app"
    if rating >= 3:
        return "Above average"
    if rating >= 2:
        return "Average"
    return "Below Average"

In [15]:
apps_df["Rating_group"] = apps_df["Rating"].apply(rating_group)

**Revenue Column**

In [16]:
apps_df["Revenue"] = apps_df["Price"] * apps_df["Installs"]

In [17]:
sia = SentimentIntensityAnalyzer()

# Polarity Scores in SIA

In [18]:
reviews_df["Sentiment_Scores"] = reviews_df["Translated_Review"].apply(lambda x : sia.polarity_scores(str(x))["compound"])

In [19]:
apps_df["Last Updated"] = pd.to_datetime(apps_df["Last Updated"],errors = "coerce")

In [20]:
apps_df["Year"] = apps_df["Last Updated"].dt.year

# Plotly

In [21]:
html_files_path = "./"
if not os.path.exists(html_files_path):
    os.makedirs(html_files_path)

In [22]:
plot_containers = ""

In [23]:
def save_plot_as_html(fig , filename , insight):
    global plot_containers
    file_path = os.path.join(html_files_path , filename)
    html_content = pio.to_html(fig , full_html = False , include_plotlyjs = "inline")
    plot_containers += f"""
    <div class = "plot-container" id = "{filename}" onclick = "openPlot('{filename}')">
        <div class = "plot"> {html_content} </div>
        <div class = "insights">{insight}</div>
    </div>
    """
    fig.write_html(file_path , full_html = False , include_plotlyjs = "inline")

In [24]:
plot_width = 400
plot_height = 300
plot_bg_color = "black"
text_color = "white"
title_font = {"size" : 16}
axis_font = {"size" : 12}

***Figure 1***

In [25]:
category_counts = apps_df["Category"].value_counts().nlargest(10)
fig1 = px.bar (
    x = category_counts.index,
    y = category_counts.values,
    labels = {'x' : 'Category' , 'y' : 'Count'},
    title = "Top Categories on PlayStore",
    color = category_counts.index,
    color_discrete_sequence = px.colors.sequential.Plasma,
    width = 400,
    height = 300
)

fig1.update_layout(
    plot_bgcolor = "black",
    paper_bgcolor = "black",
    font_color = "white",
    title_font = {'size' : 16},
    xaxis = dict(title_font = {'size' : 12}),
    yaxis = dict(title_font = {'size' : 12}),
    margin = dict(l = 10 , r = 10 , t = 30 , b = 10)
)

# fig1.update_traces(marker = dict(marker = dict(line = dict(color = "white" , width = 1))))
save_plot_as_html(fig1 , "Category Graph 1.html" , "The Top Categories apps on Playstore")

***Figure 2***

In [26]:
type_counts = apps_df["Type"].value_counts()
fig2 = px.pie (
    values = type_counts.values,
    names = type_counts.index,
    title = "App type distribution",
    color_discrete_sequence = px.colors.sequential.RdBu,
    width = 400,
    height = 300
)

fig2.update_layout(
    plot_bgcolor = "black",
    paper_bgcolor = "black",
    font_color = "white",
    title_font = {'size' : 16},
    margin = dict(l = 10 , r = 10 , t = 30 , b = 10)
)

# fig1.update_traces(marker = dict(marker = dict(line = dict(color = "white" , width = 1))))
save_plot_as_html(fig2 , "Type Graph 1.html" , "Most apps in the PlayStore are free, indicating a strategy to attract users first")

***Figure 3***

In [27]:
fig3 = px.histogram (
    apps_df,
    x = "Rating",
    nbins = 20,
    title = "Rating distribution",
    color_discrete_sequence = ['#636EFA'],
    width = 400,
    height = 300
)

fig3.update_layout(
    plot_bgcolor = "black",
    paper_bgcolor = "black",
    font_color = "white",
    title_font = {'size' : 16},
    xaxis = dict(title_font = {'size' : 12}),
    yaxis = dict(title_font = {'size' : 12}),
    margin = dict(l = 10 , r = 10 , t = 30 , b = 10)
)

# fig1.update_traces(marker = dict(marker = dict(line = dict(color = "white" , width = 1))))
save_plot_as_html(fig3 , "Rating Graph 3.html" , "Ratings are skewed towards higher values, suggesting that most apps are suggested favorably by users")

***Figure 4***

In [28]:
sentiment_counts = reviews_df["Sentiment_Scores"].value_counts()
fig4 = px.bar (
    x = sentiment_counts.index,
    y = sentiment_counts.values,
    labels = {'x' : 'Sentiment Score' , 'y' : 'Count'},
    title = "Sentiment Distribution",
    color = sentiment_counts.index,
    color_discrete_sequence = px.colors.sequential.RdPu,
    width = 400,
    height = 300
)

fig4.update_layout(
    plot_bgcolor = "black",
    paper_bgcolor = "black",
    font_color = "white",
    title_font = {'size' : 16},
    xaxis = dict(title_font = {'size' : 12}),
    yaxis = dict(title_font = {'size' : 12}),
    margin = dict(l = 10 , r = 10 , t = 30 , b = 10)
)

# fig1.update_traces(marker = dict(marker = dict(line = dict(color = "white" , width = 1))))
save_plot_as_html(fig4 , "Sentiment Graph 4.html" , "Sentiment in reviews shows mixed positive and negative feedback")

***Figure 5***

In [30]:
installs_by_category = apps_df.groupby("Category")["Installs"].sum().nlargest(10)
fig5 = px.bar (
    x = installs_by_category.index,
    y = installs_by_category.values,
    orientation = 'h',
    labels = {'x' : 'Installs' , 'y' : 'Category'},
    title = "Installs by Category",
    color = installs_by_category.index,
    color_discrete_sequence = px.colors.sequential.Blues,
    width = 400,
    height = 300
)

fig5.update_layout(
    plot_bgcolor = "black",
    paper_bgcolor = "black",
    font_color = "white",
    title_font = {'size' : 16},
    xaxis = dict(title_font = {'size' : 12}),
    yaxis = dict(title_font = {'size' : 12}),
    margin = dict(l = 10 , r = 10 , t = 30 , b = 10)
)

# fig1.update_traces(marker = dict(marker = dict(line = dict(color = "white" , width = 1))))
save_plot_as_html(fig5 , "Installs Graph 5.html" , "Categories with most Installs are Social and communication apps , reflecting their broad usage")

***Figure 6***

In [35]:
updates_per_year = apps_df["Last Updated"].dt.year.value_counts().sort_index()
fig6 = px.line (
    x = updates_per_year.index,
    y = updates_per_year.values,
    labels = {'x' : 'Year' , 'y' : 'No of Updates'},
    title = "Number Of Updates Over The Years",
    color_discrete_sequence = ['#AB63FA'],
    width = 400,
    height = 300
)

fig6.update_layout(
    plot_bgcolor = "black",
    paper_bgcolor = "black",
    font_color = "white",
    title_font = {'size' : 16},
    xaxis = dict(title_font = {'size' : 12}),
    yaxis = dict(title_font = {'size' : 12}),
    margin = dict(l = 10 , r = 10 , t = 30 , b = 10)
)

# fig1.update_traces(marker = dict(marker = dict(line = dict(color = "white" , width = 1))))
save_plot_as_html(fig6 , "Updates Graph 6.html" , "Updates have been increasing over the year, showing that developers are actively maintaining and improving their apps")

***Figure 7***

In [36]:
revenue_by_category = apps_df.groupby("Category")["Revenue"].sum().nlargest(10)
fig7 = px.bar (
    x = installs_by_category.index,
    y = installs_by_category.values,
    labels = {'x' : 'Category' , 'y' : 'Revenue'},
    title = "Revenue by Category",
    color = installs_by_category.index,
    color_discrete_sequence = px.colors.sequential.Greens,
    width = 400,
    height = 300
)

fig7.update_layout(
    plot_bgcolor = "black",
    paper_bgcolor = "black",
    font_color = "white",
    title_font = {'size' : 16},
    xaxis = dict(title_font = {'size' : 12}),
    yaxis = dict(title_font = {'size' : 12}),
    margin = dict(l = 10 , r = 10 , t = 30 , b = 10)
)

# fig1.update_traces(marker = dict(marker = dict(line = dict(color = "white" , width = 1))))
save_plot_as_html(fig7 , "Revenue Graph 7.html" , "Categories such as Business and Productivity leads in revenue")

***Figure 8***

In [37]:
genre_counts = apps_df["Category"].str.split(";" , expand = True).stack().value_counts().nlargest(10)
fig8 = px.bar (
    x = genre_counts.index,
    y = genre_counts.values,
    labels = {'x' : 'Genre' , 'y' : 'Count'},
    title = "Top Genres",
    color = installs_by_category.index,
    color_discrete_sequence = px.colors.sequential.OrRd,
    width = 800,
    height = 600
)

fig8.update_layout(
    plot_bgcolor = "black",
    paper_bgcolor = "black",
    font_color = "white",
    title_font = {'size' : 16},
    xaxis = dict(title_font = {'size' : 12}),
    yaxis = dict(title_font = {'size' : 12}),
    margin = dict(l = 10 , r = 10 , t = 30 , b = 10)
)

# fig1.update_traces(marker = dict(marker = dict(line = dict(color = "white" , width = 1))))
save_plot_as_html(fig8 , "Genre Graph 8.html" , "Actions and Casual genres are the most common ")

***Figure 9***

In [38]:
fig9=px.scatter(
    apps_df,
    x='Last Updated',
    y='Rating',
    color='Type',
    title='Impact of Last Update on Rating',
    color_discrete_sequence=px.colors.qualitative.Vivid,
    width=400,
    height=300
)
fig9.update_layout(
    plot_bgcolor='black',
    paper_bgcolor='black',
    font_color='white',
    title_font={'size':16},
    xaxis=dict(title_font={'size':12}),
    yaxis=dict(title_font={'size':12}),
    margin=dict(l=10,r=10,t=30,b=10)
)
#fig1.update_traces(marker=dict(pattern=dict(line=dict(color='white',width=1))))
save_plot_as_html(fig9,"Update Graph 9.html","The Scatter Plot shows a weak correlation between the last update and ratings, suggesting that more frequent updates dont always result in better ratings.")

***Figure 10***

In [40]:
#Figure 10
fig10=px.box(
    apps_df,
    x='Type',
    y='Rating',
    color='Type',
    title='Rating for Paid vs Free Apps',
    color_discrete_sequence=px.colors.qualitative.Pastel,
    width=400,
    height=300
)
fig10.update_layout(
    plot_bgcolor='black',
    paper_bgcolor='black',
    font_color='white',
    title_font={'size':16},
    xaxis=dict(title_font={'size':12}),
    yaxis=dict(title_font={'size':12}),
    margin=dict(l=10,r=10,t=30,b=10)
)
#fig1.update_traces(marker=dict(pattern=dict(line=dict(color='white',width=1))))
save_plot_as_html(fig10,"Paid Free Graph 10.html","Paid apps generally have higher ratings compared to free apps, suggesting that users expect higher quality from apps they pay for")

In [41]:
plot_containers_split=plot_containers.split('</div>')

In [42]:
if len(plot_containers_split) > 1:
    final_plot=plot_containers_split[-2]+'</div>'
else:
    final_plot=plot_containers

In [48]:
dashboard_html= """
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name=viewport" content="width=device-width,initial-scale-1.0">
    <title> Google Play Store Review Analytics</title>
    <style>
        body {{
            font-family: Arial, sans-serif;
            background-color: #333;
            color: #fff;
            margin: 0;
            padding: 0;
        }}
        .header {{
            display: flex;
            align-items: center;
            justify-content: center;
            padding: 20px;
            background-color: #444
        }}
        .header img {{
            margin: 0 10px;
            height: 50px;
        }}
        .container {{
            display: flex;
            flex-wrap: wrap;
            justify_content: center;
            padding: 20px;
        }}
        .plot-container {{
            border: 2px solid #555
            margin: 10px;
            padding: 10px;
            width: {plot_width}px;
            height: {plot_height}px;
            overflow: hidden;
            position: relative;
            cursor: pointer;
        }}
        .insights {{
            display: none;
            position: absolute;
            right: 10px;
            top: 10px;
            background-color: rgba(0,0,0,0.7);
            padding: 5px;
            border-radius: 5px;
            color: #fff;
        }}
        .plot-container: hover .insights {{
            display: block;
        }}
        </style>
        <script>
            function openPlot(filename) {{
                window.open(filename, '_blank');
                }}
        </script>
    </head>
    <body>
        <div class= "header">
            <img src="google_logo.webp" alt="Google Logo">
            <h1>Google Play Store Reviews Analytics</h1>
            <img src="playstore_logo.webp" alt="Google Play Store Logo">
        </div>
        <div class="container">
            {plots}
        </div>
    </body>
    </html>
    """


In [49]:
final_html=dashboard_html.format(plots=plot_containers,plot_width=plot_width,plot_height=plot_height)

In [50]:
dashboard_path=os.path.join(html_files_path,"web page.html")

In [51]:
with open(dashboard_path, "w", encoding="utf-8") as f:
    f.write(final_html)

In [52]:
webbrowser.open('file://'+os.path.realpath(dashboard_path))

True